# W05 課堂練習：三種切法比較（15 分鐘）

同一份講義（一元一次方程式），用三種方式切塊，比較：

1. 塊數、每塊的 token 數
2. 同一組問題的 Top-3 命中率

**做法**：由上往下依序執行每個儲存格（`Shift + Enter`）。第一個儲存格需下載模型，約 1 分鐘。

In [ ]:
# 1. 安裝套件、讀取講義、載入模型
!pip install -q sentence-transformers

import re, urllib.request
import numpy as np, pandas as pd
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

URL = "https://raw.githubusercontent.com/NCU-AIKslab/ncu-1151-creative-learning-final/main/data/textbook/linear-equation.md"
md = urllib.request.urlopen(URL).read().decode("utf-8")

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")  # 與地端模型相同的 tokenizer
model = SentenceTransformer("BAAI/bge-small-zh-v1.5")             # 中文嵌入模型
ntok = lambda s: len(tok.encode(s))

# 解析講義：每個 ### 標題是一個段落 id
sections, chapter, section, cur = [], None, None, None
for line in md.splitlines():
    if line.startswith("# "):
        chapter = line[2:].strip()
    elif m := re.match(r"^## (\d+-\d+) (.+)", line):
        section, cur = f"{m[1]} {m[2]}", None
    elif line.startswith("## "):
        cur = None
    elif m := re.match(r"^### (\d+-\d+-\d+) (.+)", line):
        cur = {"id": m[1], "path": f"{chapter} > {section} > {m[2]}", "text": ""}
        sections.append(cur)
    elif cur is not None:
        cur["text"] += line + "\n"
for s in sections:
    s["text"] = s["text"].strip()

print(f"講義共 {len(sections)} 個段落、{ntok(md)} tokens")

## 2. 三種切法

| 切法 | 做法 |
| --- | --- |
| 固定長度 | 全文接在一起，每 150 tokens 切一塊，不管段落邊界 |
| 依段落 | 以空行分段，只有內文 |
| 結構＋標題 | 每個 `###` 段落一塊，前面加上「章 > 節 > 標題」 |

每一塊都記錄它涵蓋了哪些段落 id，用來判斷是否命中。

In [ ]:
# 2. 切塊並量測
def chunk_fixed(size=150):
    tokens, owner = [], []
    for s in sections:
        t = tok.encode(s["text"] + "\n")
        tokens += t
        owner += [s["id"]] * len(t)
    return [{"text": tok.decode(tokens[i:i + size]), "ids": set(owner[i:i + size])}
            for i in range(0, len(tokens), size)]

def chunk_paragraph():
    return [{"text": p.strip(), "ids": {s["id"]}}
            for s in sections for p in s["text"].split("\n\n") if p.strip()]

def chunk_structure():
    return [{"text": f"{s['path']}\n{s['text']}", "ids": {s["id"]}} for s in sections]

methods = {"固定長度": chunk_fixed(), "依段落": chunk_paragraph(), "結構＋標題": chunk_structure()}

stats = []
for name, chunks in methods.items():
    n = [ntok(c["text"]) for c in chunks]
    stats.append([name, len(chunks), round(np.mean(n), 1), max(n), min(n)])
pd.DataFrame(stats, columns=["切法", "塊數", "平均 tokens", "最長", "最短"])

## 3. 檢索與命中率

8 個問題，每題標好應命中的段落 id。表格顯示涵蓋正確段落的塊排在第幾名；❌ 表示前 3 名都沒有。

In [ ]:
# 3. 建索引、檢索、算命中率
QUESTIONS = [
    ("移項的時候，符號要怎麼變？",           {"1-3-1"}),
    ("怎麼確認解出來的答案是對的？",         {"1-3-5"}),
    ("方程式的解是什麼意思？",               {"1-1-4"}),
    ("未知數在等號兩邊時怎麼解？",           {"1-3-4"}),
    ("解應用問題有哪些步驟？",               {"1-5-1"}),
    ("1-3 節有哪些例題？",                  {"1-3-3", "1-3-4"}),
    ("行程問題怎麼解？",                    {"1-5-5"}),
    ("雞和兔子共 20 隻、腳 56 隻，要怎麼列式？", {"1-5-4"}),
]
K = 3

index = {name: model.encode([c["text"] for c in chunks], normalize_embeddings=True)
         for name, chunks in methods.items()}

def search(name, q, k=K):
    qv = model.encode([q], normalize_embeddings=True)[0]
    return [methods[name][i] for i in np.argsort(-(index[name] @ qv))[:k]]

def rank(name, q, gold):
    for i, c in enumerate(search(name, q), 1):
        if c["ids"] & gold:
            return i
    return None

ranks = {name: [rank(name, q, gold) for q, gold in QUESTIONS] for name in methods}
result = pd.DataFrame({"問題": [q for q, _ in QUESTIONS],
                       **{name: [f"第 {r} 名" if r else "❌" for r in ranks[name]] for name in methods}})
result.loc[len(result)] = ["Top-1 命中率"] + [f"{sum(r == 1 for r in ranks[n]) / len(QUESTIONS):.0%}" for n in methods]
result.loc[len(result)] = ["Top-3 命中率"] + [f"{sum(r is not None for r in ranks[n]) / len(QUESTIONS):.0%}" for n in methods]
result

## 4. 看實際檢索到什麼

把 `Q` 改成上面任一題（或你自己的問題），比較三種切法的 Top-3 內容。

In [ ]:
# 4. 並排比較單一問題的檢索結果
Q = "行程問題怎麼解？"

for name in methods:
    print(f"===== {name} =====")
    for rank, c in enumerate(search(name, Q), 1):
        print(f"{rank}. {sorted(c['ids'])}  {c['text'][:60]}…".replace("\n", " "))
    print()

## 5. 回報

到[表單](【表單連結】)填寫：

1. 三種切法的 Top-1、Top-3 命中率
2. 一個「某種切法明顯比較好或比較差」的具體例子：哪一題、哪種切法、你認為的原因